# GameTheory 16j — Screening et loyer d'information

**Objectifs** : calculer les contrats, vérifier l’auto-sélection et décomposer le coût du screening.

**Prérequis** : optimisation quadratique, espérance et Python. **Durée** : 60 minutes.

**Navigation** : [Version C#](GameTheory-16j-Screening-Loyer-Information-CSharp.ipynb) | [<< 16d-Players-LLM](GameTheory-16d-LLM-Players-Othman-Sandholm-Python.ipynb) | [Index GameTheory](README.md) | [Capstone 21 -- Choix social >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb) | [jumeau Rust](GameTheory-16j-Screening-Loyer-Information-Rust.ipynb)

**Le coût mesuré de l'asymétrie d'information** dans le problème classique de *screening*
à sélection adverse (Maskin–Riley, Myerson).

À la famille **16 (Mécanismes)**, à côté de 16i (Appariage stable) : un principal
(employeur) offre un contrat à un agent (salarié) dont il **ne connaît pas le type**.
On mesure trois choses :

1. le **loyer d'information** que le principal doit verser au type haut pour que les deux
   types s'autosélectionnent honnêtement ;
2. la **distorsion** du contrat du type bas (on le fait travailler *moins* que l'optimum) —
   le type haut, lui, reste à son **premier optimum** (théorème de non-distorsion au
   sommet) ;
3. le **coût total de l'asymétrie** = surplus avec information complète − surplus avec
   screening, et sa **décomposition** en loyer + perte d'efficacité.

Toutes les valeurs sont **calculées** dans le carnet (numpy, CPU seul), pas énoncées.


## 1. Le modèle

Un salarié est de l'un de deux **types** (productivités) :

- type **bas** $\theta_L = 1$, type **haut** $\theta_H = 2$ ;
- probabilité $p$ que le salarié soit du type haut.

Le principal propose un contrat $(a, t)$ : un niveau d'effort $a$ et une rémunération $t$.

| | Formule | Rôle |
|---|---|---|
| Production | $V(\theta, a) = \theta \, a$ | ce que l'effort $a$ d'un type $\theta$ rapporte au principal |
| Coût d'effort | $C(\theta, a) = \dfrac{a^2}{2\theta}$ | ce que l'effort $a$ coûte au type $\theta$ |

Le type haut trouve l'effort **moins coûteux** ($C$ décroît en $\theta$) : c'est la condition
de **croisement simple** (single-crossing) qui rend le screening possible. L'agent a une
utilité $U(\theta) = t - C(\theta, a)$ ; le principal un surplus $S(\theta) = V(\theta, a) - t$.

> Le type est **non observable** par le principal : c'est l'asymétrie d'information.


In [1]:
import numpy as np

# --- Types et probabilité ---
theta_L = 1.0
theta_H = 2.0
p_H     = 0.5     # probabilité que le salarié soit du type haut

# --- Production et coût d'effort ---
def V(th, a):  return th * a
def C(th, a):  return a**2 / (2.0 * th)

print(f"theta_L = {theta_L}, theta_H = {theta_H}, p_H = {p_H}")
print(f"coût d'effort du type haut  C(2, a=2) = {C(theta_H, 2.0)}")
print(f"coût d'effort du type bas   C(1, a=2) = {C(theta_L, 2.0)}")
print("(le type haut trouve l'effort moins cher : croisement simple)")

theta_L = 1.0, theta_H = 2.0, p_H = 0.5
coût d'effort du type haut  C(2, a=2) = 1.0
coût d'effort du type bas   C(1, a=2) = 2.0
(le type haut trouve l'effort moins cher : croisement simple)


### Lecture

À effort égal ($a=2$), le type haut paie un coût $1$ là où le type bas en paie $2$.
C'est cette **asymétrie de coût** que le mécanisme va exploiter : en proposant le bon
contrat, on fait en sorte que chaque type *choisisse de lui-même* le sien.


## 2. Premier optimum — le principal connaît le type (benchmark)

Sans asymétrie, le principal fixe pour **chaque** type l'effort qui maximise son surplus
$S(\theta) = \theta a - a^2/(2\theta)$, et paie juste le coût (l'agent ne garde **aucun
loyer** : la contrainte de participation est liante, $U=0$).

Le maximum en $a$ donne $a^*(\theta) = \theta^2$ (dérivée nulle : $\theta - a/\theta = 0$).


In [2]:
# --- Premier optimum (information complète) ---
a_L_1F = theta_L**2
a_H_1F = theta_H**2
t_L_1F = C(theta_L, a_L_1F)          # rémunération = coût  ->  loyer d'information = 0
t_H_1F = C(theta_H, a_H_1F)
S_L_1F = V(theta_L, a_L_1F) - t_L_1F
S_H_1F = V(theta_H, a_H_1F) - t_H_1F

print("PREMIER OPTIMUM (le principal voit le type) :")
print(f"  type bas : a* = {a_L_1F:.2f},  t = {t_L_1F:.3f},  surplus = {S_L_1F:.3f}")
print(f"  type haut: a* = {a_H_1F:.2f},  t = {t_H_1F:.3f},  surplus = {S_H_1F:.3f}")
print(f"  (aucun loyer d'information : l'agent est payé à juste son coût)")

PREMIER OPTIMUM (le principal voit le type) :
  type bas : a* = 1.00,  t = 0.500,  surplus = 0.500
  type haut: a* = 4.00,  t = 4.000,  surplus = 4.000
  (aucun loyer d'information : l'agent est payé à juste son coût)


### Lecture

Avec information complète, le type haut travaille à $a^* = 4$ et le principal en retire
un surplus de **4.00**. C'est le **plafond** : tout ce qui manque plus bas est dû à
l'asymétrie. On garde ce chiffre en tête.


### Exercice 1 — Trois types

Pour trois types $\theta_L<\theta_M<\theta_H$ de probabilités positives, calculez les efforts optimaux des types bas et milieu, puis le loyer du type milieu. La participation basse et les contraintes d'incitation descendantes adjacentes sont liantes. Complétez la fonction pour différents paramètres, puis appliquez-la aux trois types $1,1.5,2$ équiprobables.

Restez dans le régime séparateur où les efforts issus des conditions du premier ordre sont croissants avec le type, comme dans les fixtures proposées. Si cet ordre échoue pour un autre prior, le regroupement de contrats et son optimisation dépassent cet exercice.

In [3]:
def trois_types(lo, milieu, haut, p_lo, p_milieu, p_haut):
    # TODO : retourner (effort bas, effort milieu, loyer milieu)
    return None

cas_trois = [(1,1.5,2,1/3,1/3,1/3,.6,1.8,.06), (1,2,3,.2,.3,.5,1/3,18/7,1/36), (.8,1.2,2,.5,.3,.2,.48,1.136842105263158,.048)]
rep = [trois_types(*c[:6]) for c in cas_trois]
if any(x is None for x in rep):
    print("Exercice 1 à compléter")
else:
    assert all(isinstance(x,tuple) and len(x)==3 and all(np.isfinite(v) for v in x) and all(abs(x[j]-c[j+6])<1e-8 for j in range(3)) for x,c in zip(rep,cas_trois))
    print("Exercice 1 : sondes validées ; cas équiprobable :", trois_types(1,1.5,2,1/3,1/3,1/3))

Exercice 1 à compléter


## 3. Second optimum — le principal ne voit pas le type (screening)

Le principal propose un **menu** de deux contrats. Les deux contraintes d'un menu
séparateur optimal :

- **Participation** du type bas liante : $U_L = 0$ (on ne paie pas de loyer au type bas) ;
- **Auto-sélection du type haut** liante : le type haut ne doit pas préférer le contrat
  du type bas — c'est cette contrainte qui crée le loyer.

Le résultat classique (croisement simple) est le **théorème de non-distorsion au
sommet** : **le type haut reçoit l'effort du premier optimum** ($a_H = \theta_H^2$,
non déformé), et **le type bas voit son effort abaissé** ($a_L < \theta_L^2$). Le loyer
d'information du type haut est alors une **fonction du contrat du type bas** :
$$U_H = C(\theta_L, a_L) - C(\theta_H, a_L) .$$

En fixant $a_H = \theta_H^2$ et en exprimant le surplus espéré du principal, son problème
se réduit au choix unique de $a_L$ :
$$\max_{a_L}\; p_H\,S_H(a_L) + (1-p_H)\,S_L(a_L)
\quad\Rightarrow\quad
a_L^{2B} = \frac{(1-p_H)\,\theta_L}{p_H\left(\frac{1}{\theta_L}-\frac{1}{\theta_H}\right) + \frac{1-p_H}{\theta_L}} .$$


In [4]:
# --- Second optimum (screening) — théorème de non-distorsion au sommet ---
# Le type HAUT n'est PAS déformé : il travaille à son premier optimum.
a_H_2B = theta_H**2                     # type haut : NON déformé (= 1er optimum)
# Le type BAS est déformé : a_L résout le FOC du surplus espéré du principal.
a_L_2B = ((1 - p_H) * theta_L) / (p_H * (1/theta_L - 1/theta_H) + (1 - p_H)/theta_L)

# Loyer d'information du type haut = fonction du contrat du type BAS
U_H = C(theta_L, a_L_2B) - C(theta_H, a_L_2B)

t_L_2B = C(theta_L, a_L_2B)             # U_L = 0 (participation liante)
t_H_2B = C(theta_H, a_H_2B) + U_H       # t_H = coût haut + loyer (auto-sélection liante)
S_L_2B = V(theta_L, a_L_2B) - t_L_2B
S_H_2B = V(theta_H, a_H_2B) - t_H_2B

print("SECOND OPTIMUM (screening, le type est caché) :")
print(f"  type haut : a = {a_H_2B:.2f}  (NON déformé = 1er optimum),  surplus = {S_H_2B:.3f}")
print(f"  type bas  : a = {a_L_2B:.3f}  (abaissé vs {a_L_1F:.2f}),   surplus = {S_L_2B:.3f}")
print(f"  LOYER D'INFORMATION du type haut  U_H = {U_H:.3f}  (= fonction de a_L)")
print(f"  rémunération type haut t_H = {t_H_2B:.3f}  (= coût {C(theta_H, a_H_2B):.3f} + loyer {U_H:.3f})")


SECOND OPTIMUM (screening, le type est caché) :
  type haut : a = 4.00  (NON déformé = 1er optimum),  surplus = 3.889
  type bas  : a = 0.667  (abaissé vs 1.00),   surplus = 0.444
  LOYER D'INFORMATION du type haut  U_H = 0.111  (= fonction de a_L)
  rémunération type haut t_H = 4.111  (= coût 4.000 + loyer 0.111)


### Lecture

Contre-intuitif : c'est le **type bas** qui est déformé (de $a^* = 1$ à $a_L = 0.67$),
pas le type haut. Le type haut travaille à son **premier optimum** $a_H = 4$ — c'est le
**théorème de non-distorsion au sommet**. En échange de ce contrat non déformé, le
principal lui verse un **loyer d'information** $U_H = 0.11$ : c'est le prix pour qu'il
ne se fasse pas passer pour le type bas. Ce loyer est une fonction du contrat du type
**bas** — réduire $a_L$ diminue la rente nécessaire mais crée une perte d'efficacité.


### Exercice 2 — Vérifier l'auto-sélection

Écrivez une fonction qui compare l'utilité d'un agent de type $\theta$ dans deux contrats $(a,t)$, puis vérifiez le type haut du menu calculé ci-dessus. Testez aussi une comparaison stricte dans chaque sens.

Pour les comparaisons numériques, traitez comme égaux deux résultats dont l’écart est inférieur à $10^{-9}$. 

In [5]:
def compare_contrats(theta, a_propre, t_propre, a_autre, t_autre):
    # TODO : retourner (utilité propre, utilité autre, propre au moins aussi bon)
    return None

cas_ic = [(theta_H,a_H_2B,t_H_2B,a_L_2B,t_L_2B), (2,2,2,1,2), (1,1,2,2,4), (2,1,2,2,2)]
attendus_ic = [(U_H,U_H,True),(1.0,1.75,False),(1.5,2.0,False),(1.75,1.0,True)]
rep_ic = [compare_contrats(*c) for c in cas_ic]
if any(x is None for x in rep_ic):
    print("Exercice 2 à compléter")
else:
    assert all(isinstance(x,tuple) and len(x)==3 and all(np.isfinite(v) for v in x[:2]) and abs(x[0]-y[0])<1e-9 and abs(x[1]-y[1])<1e-9 and isinstance(x[2],(bool,np.bool_)) and x[2]==y[2] for x,y in zip(rep_ic,attendus_ic))
    print("Exercice 2 : sondes validées")

Exercice 2 à compléter


## 4. Le coût de l'asymétrie, et sa décomposition

Le coût pour le **principal** est la perte de son surplus par rapport à l'information complète. Si le salarié est de type haut, le principal verse une rémunération supérieure au coût de l'effort : l'agent **reçoit** ce loyer. Pour le type bas, la perte provient de la distorsion de l'effort. La moyenne pondère ces deux pertes par les probabilités.

In [6]:
# --- Coût de l'asymétrie + décomposition (les deux types) ---
# Type haut : tout le coût est le loyer (son effort n'est pas déformé)
cost_H   = S_H_1F - S_H_2B
rent_H   = U_H
distort_H = 0.0                                # a_H non déformé -> perte d'efficacité nulle
# Type bas : tout le coût est la distorsion (pas de loyer, U_L = 0)
cost_L   = S_L_1F - S_L_2B
distort_L = S_L_1F - (V(theta_L, a_L_2B) - C(theta_L, a_L_2B))
rent_L    = 0.0                                # U_L = 0

print("COÛT DE L'ASYMÉTRIE (par salarié, par type) :")
print(f"  type haut : coût = {cost_H:.3f}  = loyer {rent_H:.3f} + distorsion {distort_H:.3f}")
print(f"  type bas  : coût = {cost_L:.3f}  = loyer {rent_L:.3f} + distorsion {distort_L:.3f}")
assert abs(rent_H + distort_H - cost_H) < 1e-9, "décomp. type haut incohérente"
assert abs(rent_L + distort_L - cost_L) < 1e-9, "décomp. type bas incohérente"

# Coût espéré (moyenne sur les deux types) + décomposition espérée
cost_expected    = p_H * cost_H + (1 - p_H) * cost_L
rent_expected    = p_H * rent_H + (1 - p_H) * rent_L
distort_expected = p_H * distort_H + (1 - p_H) * distort_L
print(f"  coût ESPÉRÉ (p_H = {p_H}) = {cost_expected:.3f}")
print(f"    dont loyer d'info espéré   = {rent_expected:.3f}   ({100*rent_expected/cost_expected:.0f} %)")
print(f"    dont distorsion espérée    = {distort_expected:.3f}   ({100*distort_expected/cost_expected:.0f} %)")


COÛT DE L'ASYMÉTRIE (par salarié, par type) :
  type haut : coût = 0.111  = loyer 0.111 + distorsion 0.000
  type bas  : coût = 0.056  = loyer 0.000 + distorsion 0.056
  coût ESPÉRÉ (p_H = 0.5) = 0.083
    dont loyer d'info espéré   = 0.056   (67 %)
    dont distorsion espérée    = 0.028   (33 %)


### Lecture — le résultat central

Avec $\theta_L=1$, $\theta_H=2$ et $p_H=0.5$, la perte espérée du principal est $1/12 \approx 0.083$ : loyer versé au type haut $1/18$ en moyenne et distorsion du type bas $1/36$. Le type haut **reçoit** le loyer et son effort n'est pas déformé ; le type bas ne reçoit pas de loyer mais son effort est déformé. Le partage dépend des paramètres.

## 5. Sensibilité — le coût croît avec l'écart de type

Que se passe-t-il quand les deux types sont **plus différents** ? On fixe
$\theta_L = 1$ et on fait varier $\theta_H$, en recalculant le coût **espéré** de
l'asymétrie (loyer du type haut + distorsion du type bas) pour chaque écart.


In [7]:
# --- Sensibilité : coût d'asymétrie espéré en fonction de l'écart de type ---
print(f"{'theta_H':>8} | {'a_H':>6} | {'a_L 2B':>8} | {'loyer H':>9} | {'dist. L':>9} | {'coût esp.':>10}")
print("-" * 60)
for thH in [1.25, 1.5, 2.0, 2.5, 3.0]:
    aH = thH**2                                        # type haut : 1er optimum (non déformé)
    aL = ((1 - p_H) * theta_L) / (p_H * (1/theta_L - 1/thH) + (1 - p_H)/theta_L)  # type bas : déformé
    rentH = C(theta_L, aL) - C(thH, aL)               # loyer du type haut = f(a_L)
    S1F_H = V(thH, aH) - C(thH, aH)
    S2B_H = V(thH, aH) - (C(thH, aH) + rentH)
    costH = S1F_H - S2B_H                              # = rentH (a_H non déformé)
    aL1F  = theta_L**2
    S1F_L = V(theta_L, aL1F) - C(theta_L, aL1F)
    S2B_L = V(theta_L, aL) - C(theta_L, aL)
    distL = S1F_L - S2B_L                              # distorsion du type bas
    cost_exp = p_H * costH + (1 - p_H) * distL
    print(f"{thH:>8.2f} | {aH:>6.2f} | {aL:>8.3f} | {rentH:>9.3f} | {distL:>9.3f} | {cost_exp:>10.3f}")


 theta_H |    a_H |   a_L 2B |   loyer H |   dist. L |  coût esp.
------------------------------------------------------------
    1.25 |   1.56 |    0.833 |     0.069 |     0.014 |      0.042
    1.50 |   2.25 |    0.750 |     0.094 |     0.031 |      0.063
    2.00 |   4.00 |    0.667 |     0.111 |     0.056 |      0.083
    2.50 |   6.25 |    0.625 |     0.117 |     0.070 |      0.094
    3.00 |   9.00 |    0.600 |     0.120 |     0.080 |      0.100


### Lecture

Le coût de l'asymétrie **croît de façon concave** avec l'écart de type : $\approx 0.04$
pour $\theta_H=1.25$, $0.083$ pour $\theta_H=2$, $\approx 0.10$ pour $\theta_H=3$. Plus
les types sont différents, plus le loyer à verser au type haut (et la distorsion du type
bas) sont gros — le résultat qu'on attendait, **mesuré** plutôt que supposé.


### Exercice 3 — Égalité loyer / distorsion ?

Avec $\theta_L=1$ et $p_H=0.5$, calculez les deux composantes espérées pour $\theta_H>1$. Cherchez si elles s'égalisent pour un type haut **fini**, et examinez le cas limite ; justifiez votre réponse à partir de leurs expressions. Le cas dégénéré $\theta_H=1$ mérite un traitement séparé.

Complétez aussi une fonction qui indique si une égalité existe pour un type haut fini strictement supérieur à 1 en fonction de $p_H$ ; explorez des probabilités entre 0 et 1.

In [8]:
def composantes(theta_haut):
    # TODO : retourner (loyer espéré, distorsion espérée)
    return None

def egalite_finie(p_haut):
    # TODO : existe-t-il un type haut fini strictement supérieur à 1 ?
    return None

ps = [.25,.4,.5,.6,.75,.9]
existence = [egalite_finie(p) for p in ps]
cas_comp = [(1.25,0.0347222222222222,0.00694444444444444),(2,1/18,1/36),(3,0.06,0.04)]
rep_comp = [composantes(c[0]) for c in cas_comp]
if any(x is None for x in rep_comp) or any(x is None for x in existence):
    print("Exercice 3 à compléter")
else:
    assert all(isinstance(x,tuple) and len(x)==2 and all(np.isfinite(v) for v in x) and abs(x[0]-c[1])<1e-9 and abs(x[1]-c[2])<1e-9 for x,c in zip(rep_comp,cas_comp))
    assert all(isinstance(x,(bool,np.bool_)) for x in existence) and sum(existence)==3 and all(existence[i] <= existence[i+1] for i in range(len(existence)-1))
    print("Exercice 3 : sondes validées ; justifiez le cas général séparément")

Exercice 3 à compléter


## 6. Exercices — récapitulatif

Les exercices répartis dans la leçon demandent de calculer le loyer à trois types, de vérifier l'auto-sélection et d'étudier l'égalité des composantes du coût.

## À retenir

- Le type haut **reçoit** un loyer versé par le principal ; son effort reste au premier optimum dans le modèle étudié.
- Le type bas ne reçoit pas de loyer et son effort est distordu. Pour $p_H=0.5$, le coût espéré du principal vaut $1/12$, dont $1/18$ de loyer et $1/36$ de distorsion.
- Le coût passe de $0.042$ à $0.100$ quand $\theta_H$ varie de $1.25$ à $3$ dans le tableau. Ces chiffres dépendent des hypothèses.

C'est le fondement des mécanismes de §16 (conception de mécanismes) : tout contrat qu'on propose à un agent informé doit payer ce loyer, et le concevoir, c'est minimiser ce coût.

**Trois mesures à retenir.**

- **Le coût de l'asymétrie est chiffré, pas énoncé** : avec `θ_L = 1`, `θ_H = 2` et `p_H = 0.5`, le coût espéré vaut `1/12 ≈ 0.083`, décomposé en **loyer** `1/18` (porté par le type haut) et **distorsion** `1/36` (portée par le type bas).
- **La signature du screening à croisement simple** : le type haut reste à son **premier optimum** (`a_H = 4.00`) — non-distorsion au sommet — pendant que le type bas est déformé (`a_L = 0.667` contre un optimum de 1).
- **Le coût croît de façon concave** avec l'écart de type : de `0.042` pour `θ_H = 1.25` à `0.100` pour `θ_H = 3.00`.

**Réserve de portée.** Les valeurs sont calculées sous les paramètres posés au §1 (production linéaire `V = θa`, coût d'effort `a²/(2θ)`, `p_H = 0.5`) : le partage loyer/distorsion en est une **conséquence**, pas une loi générale. La réponse à l'exercice 3 — le seuil où loyer et distorsion s'égalisent — est précisément l'endroit où ce partage bouge.

## Vérifiez votre compréhension

1. Qui paie et qui reçoit le loyer d'information ?

   <details><summary>Réponse</summary>

   C'est le **principal** (le bailleur, le monopole) qui **verse** le loyer ; c'est le **type haut de l'agent informé** qui le **reçoit**. Le loyer est le supplément de rente que le principal doit payer pour que le type haut révèle sa type : le type bas, lui, ne reçoit rien et subit la distorsion.

   </details>

2. Pourquoi le type haut reste-t-il à son premier optimum, sans distorsion ?

   <details><summary>Réponse</summary>

   C'est la condition d'auto-sélection de bord : le contrat du type haut est **non-contraignant** à l'équilibre, car le type haut n'a rien à imiter — il est le sommet de la hiérarchie. La distorsion tombe donc sur le type bas, qui doit s'écarter de son premier optimum pour que le type haut n'ait pas intérêt à le mimer.

   </details>

3. Que mesure l'exercice sur l'égalité des composantes du coût ?

   <details><summary>Réponse</summary>

   On cherche le **seuil sur `p_H`** (la probabilité du type haut) où le **loyer espéré égale la distorsion espérée** : en deçà, le coût est dominé par la distorsion ; au-dessus, par le loyer. Le partage n'est donc pas un invariant de la structure, il dépend des paramètres — d'où la réserve de portée du carnet.

   </details>

4. Le carnet annonce que le partage loyer/distorsion n'est pas 50/50. De quoi dépend-il ?

   <details><summary>Réponse</summary>

   Des **paramètres du modèle**, et le carnet le dit explicitement : `θ_L = 1`, `θ_H = 2`, `p_H = 0.5`, production linéaire et coût d'effort quadratique. Changer le prior ou l'écart de type déplace le partage, et c'est ce que mesure l'exercice 3 — le seuil où le loyer espéré égale la distorsion espérée. Ce qui **ne** dépend pas des paramètres, c'est la structure : le type haut paie toujours en loyer et n'est jamais déformé, le type bas est toujours déformé et ne paie jamais de loyer. La décomposition est exacte, sa proportion ne l'est pas.

   </details>

## Pour aller plus loin

**Carnet suivant.** [GameTheory-16k — Théorie des enchères (Myerson)](GameTheory-16k-Theorie-des-Encheres-Myerson-Python.ipynb) -- le miroir vendeur du même théorème : là où ce carnet mesure le **loyer** que le principal paie à l'agent informé, le 16k mesure la **réserve** que le vendeur impose pour ne pas payer cette rente.

**Lectures.**

- Le cadre est celui du **screening à sélection adverse** (Maskin-Riley, Myerson) : les références du carnet voisin `GameTheory-16c` listent Green et Laffont (1977) sur le bien public, Rochet (1987) sur la rationalisabilité en contexte quasi-linéaire, Saijo (1988) sur la réduction de l'espace de stratégies et Myerson et Satterthwaite (1983) sur l'efficience en échange bilatéral — le même appareillage théorique, côté agent.
- Le pendant formel de ce carnet est `GameTheory-17d-Lean-Screening-Signaling-Lean` : la formalisation en Lean du screening et du signaling.
- Maskin et Riley, *Monopoly with Incomplete Information*, RAND Journal of Economics, 1984 : modèle de screening et information privée.

**Carnets voisins.**

- [GameTheory-16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb) -- le carnet owner : conception de mécanismes, principe de révélation, et les bornes que le théorème d'impossibilité pose à toute promesse d'incitation.
- [GameTheory-17b — Asymmetric Information](GameTheory-17b-Asymmetric-Information-Python.ipynb) -- la famille voisine qui traite l'asymétrie d'information hors du cadre des mécanismes.
- [GameTheory-21 — Capstone Choix social](GameTheory-21-Capstone-Choix-Social-Python.ipynb) -- le capstone qui assemble les pièces du choix social.